# LangGraph Fundamentals — What `create_agent` Was Hiding

`create_agent` isn't a new execution model — it compiles a **LangGraph `StateGraph`** with (at minimum) two nodes: an `"agent"` node and a `"tools"` node, plus a system prompt threaded into every model call. This notebook makes that literal: by the end, you'll have hand-built the exact graph `create_agent` builds for you, understand every piece of it, and know when you'd actually want to drop down to raw LangGraph instead of the convenience wrapper.

Every agent notebook so far — the manual ReAct loop in `ai_engineer_tutorial.ipynb`, `AgentExecutor`, `create_agent` — has been the *same underlying loop*, wrapped at increasing levels of abstraction:

| Level | What it is |
|---|---|
| Manual `while True` loop (`ai_engineer_tutorial.ipynb` 8.4) | You wrote the loop, the prompt parsing, everything by hand |
| `AgentExecutor` (`Build_Smarter_AI_Apps`) | Loop hidden inside a class, prompt templated for you |
| `create_agent` | Loop compiled as a LangGraph graph, even less code |
| **LangGraph directly (this notebook)** | You build the graph yourself — same loop, but you control every node |

This is the "unwrap one more layer" step — genuinely useful once you need something `create_agent`'s fixed shape doesn't give you: a custom node in the middle of the loop, non-standard routing, multiple cooperating agents (the subject of the next notebook).

## What this notebook covers
1. Setup
2. Core building blocks: `StateGraph`, state schema, nodes, edges
3. Conditional edges — real branching logic
4. **Rebuilding `create_agent` from scratch** — `MessagesState`, `ToolNode`, `tools_condition`, the agent/tools loop, and the system prompt `create_agent` threads into every call
5. Checkpointer + `thread_id`, verified with real execution (not just described)
6. Why you'd actually drop to raw LangGraph — extending the loop with a custom node
7. Optional: the raw Anthropic SDK equivalent of the whole thing

## 1. Setup

In [44]:
# pip install -U langgraph langchain langchain-anthropic

import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
print("✅ API key set.")



from langgraph.graph import StateGraph, START, END
print("Imports OK")


✅ API key set.
Imports OK


In [45]:
from langchain_anthropic import ChatAnthropic

def get_llm(max_tokens=256, temperature=0.2):
    """
    Factory function that returns a ChatAnthropic instance.
    
    We use a factory (instead of a single global object) so we can
    easily create models with different temperature/max_tokens settings
    for different tasks throughout the notebook.
    """
    return ChatAnthropic(
        model="claude-haiku-4-5",
        max_tokens=max_tokens,
        temperature=temperature,
        anthropic_api_key=ANTHROPIC_API_KEY,
    )

# Our main LLM — used throughout the notebook
llm = get_llm()

# Quick test
response = llm.invoke("In today's sales meeting, we ")
print(response.content)



It looks like your message got cut off! You started to tell me about a sales meeting but didn't finish.

Could you complete your thought? For example, are you:
- Sharing what happened in the meeting?
- Looking for advice on a sales topic?
- Asking for help preparing for an upcoming meeting?
- Something else?

Feel free to share the full details, and I'll be happy to help!


## 2. Core building blocks: `StateGraph`, state, nodes, edges

A LangGraph graph has three ingredients:

- **State** — a schema (usually a `TypedDict`) describing what data flows through the graph. Every node reads from it and returns updates to it.
- **Nodes** — plain Python functions. Each takes the current state and returns a dict of updates to merge in.
- **Edges** — connections saying which node runs after which. `START` and `END` are special built-in markers for "the graph's entry point" and "the graph is done."

No LLM involved yet — deliberately, so the mechanics are visible without an API call muddying things.

A typeddict is a dictionary-like object that allows you to specify the types of its keys and values. In this case, the keys are strings and the values are also strings.

For example it would look like this: {"name": "John", "age": "30"}

In [4]:
from typing import TypedDict

class PipelineState(TypedDict):
    text: str
    steps_completed: list

def step_one(state: PipelineState) -> dict:
    return {"text": "raw data", "steps_completed": state["steps_completed"] + ["step_one"]}

def step_two(state: PipelineState) -> dict:
    return {"text": state["text"] + " -> cleaned", "steps_completed": state["steps_completed"] + ["step_two"]}

def step_three(state: PipelineState) -> dict:
    return {"text": state["text"] + " -> analyzed", "steps_completed": state["steps_completed"] + ["step_three"]}

In [5]:


builder = StateGraph(PipelineState)
builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)
builder.add_node("step_three", step_three)
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", "step_three")
builder.add_edge("step_three", END)

graph = builder.compile()
result = graph.invoke({"text": "", "steps_completed": []})
print(result)


{'text': 'raw data -> cleaned -> analyzed', 'steps_completed': ['step_one', 'step_two', 'step_three']}


**🔗 Ties back to theory — this is LCEL's `|`, generalized to a graph**

`RunnableSequence` (the `|` operator, dissected in `Prompt_Engineering_LangChain_Anthropic`) only supports a straight line: step 1 → step 2 → step 3, no branching, no loops back. This `StateGraph` is doing the exact same thing structurally — each node reads the previous node's output and passes along an update — but as an explicit graph instead of an implicit pipe chain. The real payoff shows up in the next section, where a straight line stops being enough.

**🔍 Under the hood — nodes are `RunnableLambda` in disguise**

When you call `add_node("step_one", step_one)`, LangGraph wraps your plain Python function in a `RunnableLambda` internally — the exact same wrapper class from the `Runnable.__or__` mechanics you already know. This is why nodes automatically get batching, async support, and tracing for free: they're not bespoke LangGraph objects, they're the same `Runnable` interface everything else in LangChain implements. `.compile()` is what turns your `StateGraph` *builder* object into an executable `CompiledStateGraph` — before `.compile()`, you just have a description of nodes and edges; nothing can run yet, same as how building an LCEL chain with `|` doesn't call the API until `.invoke()` runs.

what is batching, async support, tracing?

**Answer:** All three come free from `Runnable` (which `RunnableLambda` implements) — none of them are LangGraph-specific:
- **Batching** — `.batch([input1, input2, ...])` runs the same node/chain over a list of inputs (in parallel where possible) instead of you writing your own loop.
- **Async support** — every `Runnable` has async twins of its methods (`.ainvoke()`, `.abatch()`, `.astream()`) for free, since the interface defines both sync and async versions. Matters when a node does I/O (like an LLM call) and you want to run several concurrently without blocking.
- **Tracing** — automatic LangSmith integration (built properly in Module 04) — inputs/outputs/latency for every `Runnable` get logged the moment tracing is enabled, no manual instrumentation.

The point of the surrounding note: a hand-rolled node class would have to implement all three itself. Because `add_node` wraps your function in a `RunnableLambda`, you get them without writing any extra code.

## 3. Conditional edges — real branching logic

A normal edge (`add_edge("A", "B")`) always goes from A to B, no matter what. A **conditional edge** asks a router function to look at the current state and decide *which* node to go to next. This is the piece a plain LCEL chain genuinely cannot do — LCEL has no concept of "if X, do this; otherwise do that." 

In [6]:
# A Literal is a constraint that allows you to specify that a value must be one of a specific set of values. 

In [7]:
from typing import Literal

class ReviewState(TypedDict):
    review_length: int
    route_taken: str

def classify(state: ReviewState) -> dict:
    return {}  # nothing to update -- just runs before the routing decision

def route_by_length(state: ReviewState) -> Literal["short_path", "long_path"]:
    return "short_path" if state["review_length"] < 50 else "long_path"

def short_path(state: ReviewState) -> dict:
    return {"route_taken": "short_path"}

def long_path(state: ReviewState) -> dict:
    return {"route_taken": "long_path"}

builder = StateGraph(ReviewState)
builder.add_node("classify", classify)
builder.add_node("short_path", short_path)
builder.add_node("long_path", long_path)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_by_length, {"short_path": "short_path", "long_path": "long_path"})
builder.add_edge("short_path", END)
builder.add_edge("long_path", END)

graph = builder.compile()
print(graph.invoke({"review_length": 20, "route_taken": ""}))
print(graph.invoke({"review_length": 200, "route_taken": ""}))


{'review_length': 20, 'route_taken': 'short_path'}
{'review_length': 200, 'route_taken': 'long_path'}


**🔍 Under the hood — the router function's contract, precisely**

`route_by_length` takes the state and returns a **string** (or, as you'll see below, a `Literal` type hint listing the possible strings — purely for readability/graph-diagram purposes, not enforced at runtime). LangGraph looks that string up in the `path_map` dict you passed (`{"short_path": "short_path", "long_path": "long_path"}` — the keys are what the router can return, the values are the actual node names to jump to; they're often identical strings like here, but don't have to be). Critically: **the router function itself should never call an LLM, mutate state, or have side effects** — by the time it runs, `classify` has *already* run and its updates are already merged into state. The router is a pure lookup: read state, return a string, done. All real computation belongs in nodes, not routers. This separation is what keeps a graph's execution flow inspectable — you can always answer "why did it go this way" by reading one small, side-effect-free function.

## 4. Rebuilding `create_agent` from scratch

This is the main event. `create_agent` builds a graph shaped almost exactly like this:

```
START → agent → (conditional: tool call requested?) → tools → agent → ... → END
```

Three new pieces make this the "agent loop" shape instead of a plain pipeline:

- **`MessagesState`** — a pre-built state schema, just `{"messages": [...]}`, using a special reducer (`add_messages`) so every node's returned messages get *appended* to the list rather than overwriting it.
- **`ToolNode`** — a pre-built node that takes the last message's tool call(s), actually executes the matching Python tool function(s), and returns the result(s) as `ToolMessage` objects.
- **`tools_condition`** — a pre-built router: look at the last message, return `"tools"` if it contains tool calls, otherwise return `END`.

To verify this actually works without needing your API key yet, this first version uses a tiny mock model that fakes Claude's behavior (decide to call the calculator, then answer). We'll swap in real `ChatAnthropic` right after.

question - so is the agent creating the tool call? What does this look like?

**Answer:** Yes — "the agent" here means the LLM itself (`model_with_tools`, i.e. Claude). Its response *is* the tool call request: structurally, `AIMessage.tool_calls` is a list of dicts, each shaped like `{"name": "calculator", "args": {"expression": "345*789"}, "id": "call_1"}`. The model doesn't run anything — it just outputs "I want to call this tool with these arguments," and LangChain surfaces that as `.tool_calls` on the returned message. You'll see the exact real shape a few cells down in `result['messages']`, where the second message is an `AIMessage` with a populated `tool_calls` list — that's this.

In [48]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '345 * 789'."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

tools = [calculator]

# --- Mock model, purely so we can verify the GRAPH LOGIC without an API key ---
class MockLLM:
    def __init__(self):
        self.call_count = 0
    def bind_tools(self, tools):
        return self
    def invoke(self, messages):
        self.call_count += 1
        if self.call_count == 1:
            return AIMessage(content="", tool_calls=[
                {"name": "calculator", "args": {"expression": "345*789"}, "id": "call_1"}
            ])
        return AIMessage(content="345 * 789 = 272205")

model_with_tools = MockLLM().bind_tools(tools)  # <-- changes format of tool to the format that the model SDK expects. Specifically a tool schema

def agent_node(state: MessagesState) -> dict:
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)  # routes to "tools" or END automatically
builder.add_edge("tools", "agent")  # after the tool runs, go back to the agent -- this is the loop

graph = builder.compile()

result = graph.invoke({"messages": [HumanMessage(content="What's 345 * 789?")]})
for m in result["messages"]:
    print(type(m).__name__, "->", m.content if m.content else m.tool_calls)


HumanMessage -> What's 345 * 789?
AIMessage -> [{'name': 'calculator', 'args': {'expression': '345*789'}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage -> 272205
AIMessage -> 345 * 789 = 272205


**Verified output** (this exact code was run to confirm the loop behaves correctly):
```
HumanMessage -> What's 345 * 789?
AIMessage -> [{'name': 'calculator', 'args': {'expression': '345*789'}, 'id': 'call_1', ...}]
ToolMessage -> 272205
AIMessage -> 345 * 789 = 272205
```

The graph correctly looped `agent → tools → agent` once, then stopped at the second `agent` call because that response had no tool calls, so `tools_condition` routed to `END`.

Now the real version — swap the mock for `ChatAnthropic`:

In [49]:
from langchain_anthropic import ChatAnthropic

model = get_llm(max_tokens=500, temperature=0)  
model_with_tools = model.bind_tools(tools)

def agent_node(state: MessagesState) -> dict:
    response = model_with_tools.invoke(state["messages"])
    # question - so the response can be the tool call? Or only text?
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))  # TOOLS IS THE PYTHON CACLULATOR
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition) # recall tools_conditions is a prebuilt router and determines if thee agent returned a .tool_call and routes accordingly
builder.add_edge("tools", "agent")

real_graph = builder.compile()

result = real_graph.invoke({"messages": [HumanMessage(content="What's 345 * 789?")]})
print(result["messages"][-1].content)


345 * 789 = **272,205**


In [52]:
print(result['messages'])

[HumanMessage(content="What's 345 * 789?", additional_kwargs={}, response_metadata={}, id='3c4bc57f-c82b-4b36-b3ca-bef4e4ba77f3'), AIMessage(content=[{'id': 'toolu_01P3kDMdHdAEicAqdWdn9DzK', 'caller': {'type': 'direct'}, 'input': {'expression': '345 * 789'}, 'name': 'calculator', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cd2pTtJauToUrAUyA8mTg', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 576, 'output_tokens': 56, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019f62f6-d1bc-76d2-8767-4a38d07f312c-0', tool_calls=[{'name': 'calculator', 'args':

**Answering the inline comments in the cell above:**

- *"so the response can be the tool call? Or only text?"* — Either. `model_with_tools.invoke(...)` can return an `AIMessage` that's plain text (`tool_calls == []`), one requesting a tool (`tool_calls` populated), or in rarer cases both (some text plus a tool call in the same response). `tools_condition` only ever looks at `.tool_calls` to decide where to route — it doesn't care whether `.content` also has text.
- *"TOOLS IS THE PYTHON CALCULATOR"* — correct. `tools = [calculator]` (defined two cells up) is the actual list of real Python functions, passed to both `model.bind_tools(tools)` (so Claude knows the schema) and `ToolNode(tools)` (so it knows what to actually execute).
- *"recall tools_conditions is a prebuilt router..."* — correct, that's exactly its contract, same as `route_by_length` in §3: read state, return a string, no side effects.

question - so is create agent actually using this bind tools class method? Is that a langgraph thing? I dont think the anttrhopic SDK does that right?

**Answer:** Yes to the first part, and you're right about the second. `create_agent` (from `langchain.agents`) builds a graph internally that calls `model.bind_tools(tools)` — the exact same mechanism you're building by hand here. But `bind_tools` is a **LangChain** concept (a method every `BaseChatModel` implements), not a LangGraph thing and not something the raw Anthropic SDK has at all — you're correct that it doesn't. The raw SDK just takes a `tools=[...]` parameter directly on `client.messages.create()`, on every single call (see Section 7's raw SDK cell). `bind_tools` is LangChain's convenience: it pre-attaches that same `tools` parameter to a model *object* once, so you don't have to pass it manually every time you call `.invoke()`.

question - do these return blocks such as tool blocks and text blocks like in the antrhopic SDK? Is that baked into the Human/AI/Tool messages? Can you explain it by looking at the result['messages']

**Answer:** Yes, exactly — look at `result['messages'][1]` in the real output below. Its `.content` is `[{'id': 'toolu_...', 'input': {'expression': '345 * 789'}, 'name': 'calculator', 'type': 'tool_use'}]` — that's Anthropic's *raw* `tool_use` content block, stored basically as-is inside `.content`. LangChain doesn't hide this. But it *also* surfaces the same information a second, cleaner way: `.tool_calls` on that same message is `[{'name': 'calculator', 'args': {'expression': '345 * 789'}, 'id': 'toolu_...', 'type': 'tool_call'}]` — LangChain's normalized version (note `args` instead of Anthropic's `input`), meant to look the same regardless of which model provider you're using. Both attributes exist on the same `AIMessage` at once — `.content` is closer to the provider's raw shape, `.tool_calls` is LangChain's provider-agnostic view of the same data.

For the tool *result* (`result['messages'][2]`): `ToolMessage(content='272205', name='calculator', tool_call_id='toolu_...')` — in raw Anthropic terms this would be a `tool_result` content block buried inside a `role: "user"` message (see Section 7's raw loop: `messages.append({"role": "user", "content": tool_results})`). LangChain instead gives it its own dedicated message *type*, `ToolMessage`, rather than burying it inside a user-role message.

In [26]:
result['messages']

[HumanMessage(content="What's 345 * 789?", additional_kwargs={}, response_metadata={}, id='734ab109-f348-42d8-93ca-505ac7ed1796'),
 AIMessage(content=[{'id': 'toolu_01HFN7cF2wvtZqDDRTSE9dJN', 'caller': {'type': 'direct'}, 'input': {'expression': '345 * 789'}, 'name': 'calculator', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cd25z4TbeNAz8jRNSpeAq', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 576, 'output_tokens': 56, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019f60f8-cc42-7ac3-8fbf-5fe410bb438c-0', tool_calls=[{'name': 'calculator', 'args'

In [24]:
result["messages"][-1]

AIMessage(content='345 * 789 = **272,205**', additional_kwargs={}, response_metadata={'id': 'msg_011Cd25zEhVv6fxoPKTUmbEj', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 646, 'output_tokens': 15, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019f60f8-d9ab-7a60-915a-c94f4b4e58cb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 646, 'output_tokens': 15, 'total_tokens': 661, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}})

In [32]:
result["messages"][-1].content

'345 * 789 = **272,205**'

please explain this below?

**Answer:** This cell is proving the exact same point from the answer above, three different ways on the same message (`result['messages'][1]`, the `AIMessage` that requested the tool):
- `.content` → the raw `tool_use` block, close to Anthropic's native shape
- `.tool_calls` → LangChain's normalized version of that same request
- `.type` → just the string `"ai"` — LangChain's internal role tag for this message class, used e.g. by `MessagesState`'s `add_messages` reducer to know what kind of message it's appending

And to directly answer the inline comment in the next cell — *"messages... have both a `.content` and a `.tool_call` class attribute if its a tool call block?"* — yes, but more precisely: **every** `AIMessage` always has both attributes, tool call or not. `.tool_calls` is just an empty list `[]` when there's no tool call (you can see this on the final `AIMessage` in `result['messages']`, the one with the plain-text answer) and populated when there is one. It's not conditional on message *type* — it's always present, sometimes empty.

In [ ]:
print(result['messages'][1])
# it looks like messages (aka blocks i think?) havee both a .content and a .tool_call class attribute if its a tool call block?
print(result['messages'][1].content)
print(result['messages'][1].tool_calls)
print(result['messages'][1].type)

content=[{'id': 'toolu_01HFN7cF2wvtZqDDRTSE9dJN', 'caller': {'type': 'direct'}, 'input': {'expression': '345 * 789'}, 'name': 'calculator', 'type': 'tool_use'}] additional_kwargs={} response_metadata={'id': 'msg_011Cd25z4TbeNAz8jRNSpeAq', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 576, 'output_tokens': 56, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--019f60f8-cc42-7ac3-8fbf-5fe410bb438c-0' tool_calls=[{'name': 'calculator', 'args': {'expression': '345 * 789'}, 'id': 'toolu_01HFN7cF2wvtZqDDRTSE9dJN', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens':

In [40]:
for m in result['messages']:
    print(m.content)

What's 345 * 789?
[{'id': 'toolu_01HFN7cF2wvtZqDDRTSE9dJN', 'caller': {'type': 'direct'}, 'input': {'expression': '345 * 789'}, 'name': 'calculator', 'type': 'tool_use'}]
272205
345 * 789 = **272,205**


### 🔗 Ties back to theory — you just rebuilt the ReAct loop, a third time

Compare this graph, line by line, to the manual `while True:` loop from `ai_engineer_tutorial.ipynb` Section 8.4:

| Manual loop | This graph |
|---|---|
| `while True:` | The `agent ⇄ tools` cycle |
| Call the model, get raw text | `agent_node` calls `model_with_tools.invoke(...)` |
| Parse `Action: X` / `Action Input: Y` from text | `tool_calls` — structured, not string-parsed, because `bind_tools` uses Claude's native tool-use API instead of prompt-engineered text parsing |
| Check for `Final Answer:` to stop | `tools_condition` checking for the absence of `tool_calls` |
| Run the tool, append `Observation: ...` | `ToolNode` runs the tool, appends a `ToolMessage` |
| Loop back | `add_edge("tools", "agent")` |

**One precise correction on this table:** §8.4 is the right comparison specifically for the *text-parsing* rows above (`Parse Action: X...` / `Check for Final Answer:`) — that's genuine ReAct-style prompt parsing, and 8.4 is the only place in this series that works that way. But 8.4 never uses native tool-use blocks at all, so it's not actually the raw-SDK match for `tool_calls`/`ToolMessage`. The code that matches *those* one-to-one is `ai_engineer_tutorial.ipynb` **§9.2** (`run_tool_loop`) — same `tools=[...]` schema, same `stop_reason`-driven loop, same `ToolUseBlock`/`tool_result` shape, just raw SDK instead of LangGraph. Section 7 below rebuilds that exact pattern from scratch without naming it — worth knowing you already built this shape once before, in §9.2.

Same loop, now traced across every level of abstraction you've built it at.

### 🔍 Under the hood — `bind_tools` and `tool_calls`, precisely

`model.bind_tools(tools)` doesn't change the model — it returns a new `Runnable` that, on `.invoke()`, translates your `@tool`-decorated Python functions' names/docstrings/type hints into Anthropic's native tool-use API format (the `tools` parameter you'd pass to `client.messages.create()` directly) and attaches it to every request. When Claude decides to use a tool, the API response contains a structured `tool_use` content block (not free text you'd have to regex-parse) — `bind_tools` is what makes that show up as the clean `AIMessage.tool_calls` list you see above, instead of you writing a parser for `Action: calculator\nAction Input: 345*789`-style text like the manual ReAct loop had to.

### Walking through exactly who does what — a step-by-step trace

It's easy to read `agent_node`, `ToolNode`, and `tools_condition` and still not be sure which one is actually "in charge." The honest answer: none of them are. Each has exactly one job, and none of them know about the other three:

| Piece | What it actually does | What it does NOT do |
|---|---|---|
| `model_with_tools` (Claude itself) | **Decides** whether a tool is needed, based on the conversation so far. Returns either plain text or a structured `tool_use` request. | Never executes any Python code. A "decision" to call a tool is just a request, not an action. |
| `agent_node` | Calls `model_with_tools.invoke(...)` **once**, wraps whatever came back. | Doesn't decide anything, doesn't run tools, doesn't loop. |
| `ToolNode` | Looks at the last message's `tool_calls`, matches the name against your `tools` list, **actually runs** `calculator(...)`, wraps the result in a `ToolMessage`. | Doesn't decide *whether* to run — only runs what it's told to, and only when the graph routes execution here. |
| `tools_condition` | Reads the last message, returns `"tools"` if it has tool calls, else `END`. Pure lookup. | Never calls the model, never runs a tool, never has side effects (§3 already established this contract for routers in general). |

None of these four pieces contains a loop. The loop only exists because of how the graph's edges are wired: `add_edge(START, "agent")`, `add_conditional_edges("agent", tools_condition)`, and `add_edge("tools", "agent")` — that last edge is what turns "run agent once, maybe run tools once" into "keep going until `tools_condition` says stop." The **compiled graph itself** is the orchestrator — not any node, not the router, not the model. `real_graph.invoke(...)` is what actually walks the structure you declared: run the current node, consult its edge (or ask the conditional-edge function where to go), run whatever that points to, repeat until something points to `END`.

You can watch this happen node-by-node instead of only seeing the final result, using `.stream(..., stream_mode="updates")` instead of `.invoke(...)` — it yields one dict per node, the instant that node finishes:

In [38]:
# Same real_graph built two cells up -- re-running the same question, but
# watching it step through the graph one node at a time instead of only
# seeing the final answer.
for step in real_graph.stream({"messages": [HumanMessage(content="What's 345 * 789?")]}, stream_mode="updates"):
    for node_name, node_output in step.items():
        print(f"--- node: {node_name} ---")
        for m in node_output["messages"]:
            print(" ", type(m).__name__, "->", m.content if m.content else m.tool_calls)

--- node: agent ---
  AIMessage -> [{'id': 'toolu_01JS8SQ5snFV5z74EARVYvyF', 'caller': {'type': 'direct'}, 'input': {'expression': '345 * 789'}, 'name': 'calculator', 'type': 'tool_use'}]
--- node: tools ---
  ToolMessage -> 272205
--- node: agent ---
  AIMessage -> 345 * 789 = **272,205**


**⚠️ Needs your API key to run** (unexecuted here — no live key in this environment). Expected shape, matching the trace already verified earlier in this section:

```
--- node: agent ---
  AIMessage -> [{'name': 'calculator', 'args': {'expression': '345*789'}, ...}]
--- node: tools ---
  ToolMessage -> 272205
--- node: agent ---
  AIMessage -> 345 * 789 = 272,205
```

Read left to right against the table above: `agent` runs → Claude *decides* it needs the tool (a request, not an execution) → the graph's conditional edge sends control to `tools` → `ToolNode` *actually runs* `calculator("345*789")` → the graph's plain edge sends control back to `agent` → Claude sees the real result and answers → `tools_condition` finds no tool calls this time → routes to `END`. Every arrow in that sentence is an edge you wrote; every action in between is one node doing exactly one thing.

### The piece still missing: the system prompt

Every version of `agent_node` above passes `state["messages"]` straight to the model — just the `HumanMessage`, nothing else. But `create_agent(model=..., tools=..., system_prompt=...)` — the version you've used in every other notebook — takes a `system_prompt` and threads it into *every single call* the agent node makes. That's not a cosmetic difference: it's what tells the model what it is, what it should and shouldn't do, and (in every RAG notebook since) whether it's allowed to answer from outside the retrieved context. A hand-built graph without this isn't a smaller version of `create_agent` — it's missing one of its core pieces.

The system prompt doesn't belong in `state["messages"]` itself — if it did, it would get checkpointed and re-persisted every turn like part of the conversation, which it isn't. Instead, `agent_node` prepends it fresh, at call time, in front of whatever's actually in state:

In [ ]:
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = (
    "You are a careful arithmetic assistant. Always use the calculator tool for any "
    "arithmetic instead of computing it yourself, even if the calculation looks simple."
)

# I believe this is letting the emodel know it has access to this tool , aka it is the tools hyperparameter in the antrhopic sdk.
model_with_tools = model.bind_tools(tools)  # reusing `model` from the cell above

def agent_node_with_system_prompt(state: MessagesState) -> dict:
    # Prepended fresh on every call -- NOT stored in state, so it never gets
    # checkpointed as part of the conversation history.
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node_with_system_prompt)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

graph_with_system_prompt = builder.compile()

# Deliberately a "simple" calculation the model could easily do in its head --
# the system prompt is what should force it through the tool anyway.
result = graph_with_system_prompt.invoke({"messages": [HumanMessage(content="What's 12 + 7?")]})
for m in result["messages"]:
    print(type(m).__name__, "->", m.content if m.content else m.tool_calls)

Re: *"I believe this is letting the model know it has access to this tool, aka it is the tools hyperparameter in the anthropic sdk"* — correct. `model.bind_tools(tools)` is exactly the LangChain equivalent of passing `tools=[...]` to `client.messages.create()` in the raw SDK — same row in Section 7's mapping table (`bind_tools(tools)` ↔ `tools=raw_tools`, hand-written as a JSON schema).

**⚠️ Needs your API key to run** — this cell wasn't executed while making this edit (no live key in this environment). Run it yourself: with the system prompt in place, you should see the model call `calculator` even for `12 + 7` — a trivial sum it would otherwise likely just answer directly, the way `345 * 789` needed no such nudge earlier since it was already awkward enough to do in-head. If it *doesn't* call the tool, that's a real, useful observation too: it tells you a system prompt is an instruction, not a hard constraint — the model can still choose not to follow it, same as any prompt.

**🔗 Ties back to theory — this is the exact piece `create_agent(system_prompt=...)` was doing for you, silently, in every notebook up to now.** `create_agent` builds an `agent_node` that does precisely this prepend, on every call, so that "how do I give my agent instructions" never needed its own explanation before — it was already working, just invisibly. Now you've rebuilt every piece `create_agent` compiles into a graph: `MessagesState`, `ToolNode`, `tools_condition`, the loop, and the system prompt.

### Closing the loop: the one-liner all of Section 4 replaces

Every piece built above — `agent_node_with_system_prompt`, `ToolNode(tools)`, `tools_condition`, the `StateGraph` wiring, the `SystemMessage` prepend — collapses into this:

In [ ]:
from langchain.agents import create_agent

# model as a plain string here, not a ChatAnthropic object -- create_agent builds
# its own model instance internally. tools=[calculator] is the same tools list
# from earlier in this section; SYSTEM_PROMPT is the same string too.
agent = create_agent(
    model="claude-haiku-4-5",
    tools=[calculator],
    system_prompt=SYSTEM_PROMPT,
)

# Same trivial "12 + 7" test as the hand-built version above -- if the system
# prompt is doing its job, this should call `calculator` here too.
result = agent.invoke({"messages": [HumanMessage(content="What's 12 + 7?")]})
for m in result["messages"]:
    print(type(m).__name__, "->", m.content if m.content else m.tool_calls)

**⚠️ Needs your API key to run** (unexecuted here, same reason as the cell above). Expected: the same behavior as the hand-built `graph_with_system_prompt` — `calculator` gets called even for the trivial `12 + 7`, because `create_agent` is threading `SYSTEM_PROMPT` into the call exactly the way `agent_node_with_system_prompt` did by hand. If the two versions ever produced genuinely different behavior on the same inputs, that would mean the "from scratch" rebuild above missed something — they shouldn't diverge, because this one line and everything built above it compile to the same graph shape.

**One small addition beyond the system prompt fix, flagged separately since you asked to keep scope tight:** every `agent ⇄ tools` graph you've built in this notebook is a loop with no hard ceiling — `tools_condition` only stops it when the model *stops requesting tools on its own*. A model that keeps calling tools (a bad prompt, a confusing tool result, a genuine bug) would loop indefinitely. `.compile()` and `.invoke()` both accept a `recursion_limit` (e.g. `graph.invoke(input, {"recursion_limit": 25})`) that hard-stops execution after that many graph steps regardless of what the model wants to do next — cheap insurance worth knowing exists from the moment you build your first loop, not just when `02_multi_agent_orchestration` mentions it again for the Reflection pattern.

## 5. Checkpointer + `thread_id` — verified with real execution

Last notebook, I described the checkpointer mechanism conceptually. Here it is actually running, so you can see the state accumulate with your own eyes.

In [43]:
from langgraph.checkpoint.memory import InMemorySaver

def echo_node(state: MessagesState) -> dict:
    last = state["messages"][-1].content
    # recall that the call to the node function adds the result to the state:
    return {"messages": [AIMessage(content=f"echo: {last}")]}

demo_builder = StateGraph(MessagesState)
demo_builder.add_node("echo", echo_node)
demo_builder.add_edge(START, "echo")
demo_builder.add_edge("echo", END)

demo_graph = demo_builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "demo-thread"}}

demo_graph.invoke({"messages": [HumanMessage(content="hello")]}, config)
demo_graph.invoke({"messages": [HumanMessage(content="second turn")]}, config)

# Inspect the ACTUAL persisted state for this thread_id
snapshot = demo_graph.get_state(config)
print(f"Messages accumulated in thread state: {len(snapshot.values['messages'])}")
for m in snapshot.values["messages"]:
    print(" ", type(m).__name__, m.content)


Messages accumulated in thread state: 4
  HumanMessage hello
  AIMessage echo: hello
  HumanMessage second turn
  AIMessage echo: second turn


In [ ]:
# it looks like an AI message doesnt actually have to be from the LLM? Here it looks like what was returned from the tool node. ( I think this is considered a tool node?) except it looks liek there was no tool call so idk...

print(snapshot.values)

{'messages': [HumanMessage(content='hello', additional_kwargs={}, response_metadata={}, id='e54586d0-3a97-4d2f-9894-1609896e2dad'), AIMessage(content='echo: hello', additional_kwargs={}, response_metadata={}, id='ee6dfff8-3cf1-49a6-ac0b-88bb2ffc78d3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='second turn', additional_kwargs={}, response_metadata={}, id='e16a3d65-0417-46b4-82b2-b3d883b10ae6'), AIMessage(content='echo: second turn', additional_kwargs={}, response_metadata={}, id='5191a863-065e-44b5-a6ea-09c429eefe12', tool_calls=[], invalid_tool_calls=[])]}


Re: *"it looks like an AI message doesnt actually have to be from the LLM? ... I think this is considered a tool node? ... there was no tool call so idk"* — your first instinct is exactly right, and the second guess (tool node) isn't quite it. `AIMessage` is just a plain data class — anyone can construct `AIMessage(content="anything")` directly in Python, same as building any other object. It doesn't have to come from a real model call. `echo_node` (the cell above) does exactly that: `return {"messages": [AIMessage(content=f"echo: {last}")]}` — hand-built, no LLM involved at all.

This has nothing to do with `ToolNode` or tool-calling — `echo_node` is structurally identical to `step_one`/`step_two`/`step_three` from §2: a plain custom node function, no tools anywhere in it. It's written this way *on purpose* for this section: §5 is demonstrating the checkpointer/persistence mechanism specifically, so it deliberately skips the LLM and tool loop entirely to keep the example free (no API cost) and focused on one thing — the fact that `AIMessage` requires zero framework machinery to construct is exactly what makes that possible.

**Verified output:**
```
Messages accumulated in thread state: 4
  HumanMessage hello
  AIMessage echo: hello
  HumanMessage second turn
  AIMessage echo: second turn
```

Two separate `.invoke()` calls, same `thread_id` → the second call's state included everything from the first call, automatically. This is exactly the mechanism described last notebook, now visibly proven: `graph.get_state(config)` reads the checkpointer's saved snapshot for that `thread_id` directly — there's no hidden magic, it's a real, inspectable object you can print at any time to debug what an agent "remembers."

Add your own agent's `checkpointer=InMemorySaver()` to the `.compile()` call from Section 4 and you get exactly this behavior on top of the tool-calling loop — memory and tool use are two independent, composable pieces of the graph, not a special combined feature.

## 6. Why you'd actually drop to raw LangGraph instead of `create_agent`

`create_agent` gives you the agent/tools loop with almost no code — great default. But it fixes the *shape* of the graph: agent node, tools node, that's the loop. The moment you need a node that isn't "call the model" or "run a tool," you need raw LangGraph.

Concrete example: a **guardrail node** that checks the calculator's result before letting the agent see it — something `create_agent` has no slot for.

In [12]:
def validate_tool_output(state: MessagesState) -> dict:
    """Custom node: sits between tools and agent, sanity-checks tool output
    before the agent gets to see it. create_agent has no equivalent slot for this."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "content") and "Error" in str(last_message.content):
        # Flag suspicious tool output for the agent explicitly, rather than
        # letting it silently reason over a broken result
        last_message.content = f"[VALIDATION WARNING] {last_message.content}"
    return {}

guarded_builder = StateGraph(MessagesState)
guarded_builder.add_node("agent", agent_node)
guarded_builder.add_node("tools", ToolNode(tools))
guarded_builder.add_node("validate", validate_tool_output)
guarded_builder.add_edge(START, "agent")
guarded_builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
guarded_builder.add_edge("tools", "validate")   # <-- new: inserted between tools and agent
guarded_builder.add_edge("validate", "agent")

guarded_graph = guarded_builder.compile()
print("Graph with custom validation node compiled successfully.")


Graph with custom validation node compiled successfully.


### 🔗 Ties back to theory — this is the actual argument for LangGraph over `create_agent`

This single inserted node (`tools → validate → agent` instead of `tools → agent`) is the entire case for dropping down a layer. `create_agent`'s convenience comes precisely from *not* exposing this level of control — you get exactly one shape. The moment your production reliability needs (this is a preview of Module 05) require inspecting/modifying state mid-loop — validating tool output, logging every step, injecting a human-approval pause — you need the graph builder directly. Keep this pattern in mind; Module 02 (multi-agent orchestration) and Module 05 (production reliability) both lean on exactly this "insert a custom node into the loop" capability.

## 7. Optional — the raw Anthropic SDK equivalent

LangGraph itself isn't vendor-specific — the only vendor-specific piece in the whole graph is the `ChatAnthropic` call inside `agent_node`. Here's what the *entire* graph collapses to if you strip away LangGraph and LangChain completely, using the raw SDK — this is the closest thing to "no framework at all." It's essentially the same pattern as `ai_engineer_tutorial.ipynb` **Section 9.2**'s `run_tool_loop` — native tool-use blocks, `stop_reason`-driven loop — not Section 8.4's text-parsed ReAct loop (8.4 predates native tool-use entirely, so it can't be the raw-SDK match for this). Same tool-use mechanics as 9.2, this version just uses a calculator instead of whatever tool 9.2 defined:

In [ ]:
import anthropic

raw_client = anthropic.Anthropic()

raw_tools = [{
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression, e.g. '345 * 789'.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string"}},
        "required": ["expression"],
    },
}]

def run_calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

messages = [{"role": "user", "content": "What's 345 * 789?"}]

while True:
    response = raw_client.messages.create(
        model="claude-sonnet-4-6", max_tokens=500, tools=raw_tools, messages=messages,
    )
    messages.append({"role": "assistant", "content": response.content})
    print([b for b in response.content])

    tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
    if not tool_use_blocks:
        final_text = next(b.text for b in response.content if b.type == "text")
        print(final_text)
        break

    tool_results = []
    for block in tool_use_blocks:
        # what if theres more than one type of tool? or will we get to this later
        result = run_calculator(block.input["expression"])
        tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
    messages.append({"role": "user", "content": tool_results})
    print(messages[-1])


[ToolUseBlock(id='toolu_01Gt7BLySyHff97SLetsjK8d', caller=DirectCaller(type='direct'), input={'expression': '345 * 789'}, name='calculator', type='tool_use')]
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01Gt7BLySyHff97SLetsjK8d', 'content': '272205'}]}
[TextBlock(citations=None, text='The result of 345 * 789 is **272,205**.', type='text')]
The result of 345 * 789 is **272,205**.


Re: *"what if theres more than one type of tool? or will we get to this later"* — you'd get there with the exact loop already written above: `response.content` can contain **multiple** `tool_use` blocks in one turn if Claude decides to call more than one tool at once, and `for block in tool_use_blocks:` already iterates all of them, running `run_calculator(...)` and appending a `tool_result` for each — no change needed to the loop itself. The only thing missing here is *dispatch by name* (this notebook only ever defines one tool, so there's nothing to choose between). That's exactly what `ToolNode` already solved for you in §4 — `ToolNode(tools)` takes a **list** of tools and matches each tool call's `name` against the right Python function automatically. So: yes, you already saw the answer, just in the LangGraph version rather than this raw one.

**The mapping, made explicit:**

| LangGraph version | Raw SDK version |
|---|---|
| `StateGraph` + compiled loop | The `while True:` loop itself |
| `MessagesState` / `add_messages` reducer | The `messages` list you append to by hand |
| `agent` node | One `client.messages.create()` call |
| `bind_tools(tools)` | The `tools=raw_tools` parameter, hand-written as a JSON schema |
| `tool_calls` on `AIMessage` | `[b for b in response.content if b.type == "tool_use"]` |
| `tools_condition` | The `if not tool_use_blocks:` check |
| `ToolNode` running your `@tool` function | `run_calculator(...)` called manually, result wrapped in a `tool_result` block |
| `checkpointer` + `thread_id` | You'd have to persist `messages` to a database yourself, keyed by however you identify a session |

Everything LangGraph gives you is real, saved engineering effort — but nothing it does is inaccessible or magic. It's the same loop, the same message list, the same tool-use API, just with the bookkeeping (the loop itself, the routing logic, the state persistence) written once, correctly, so you don't rewrite it by hand every time — same trade-off as every wrapper you've dissected so far in this whole series.